# 00. Prerequisites: Dataset & Model Setup

Run this section **once** before starting the annotation pipeline.

**Requirements:**
- `.env` file with `KAGGLE_USERNAME` and `KAGGLE_KEY` (for dataset download)
- Python dependencies installed (`kaggle`, `ultralytics`)

In [ ]:
import sys
import os
from pathlib import Path

# ============================================================
# DJANGO SETUP — Required to use centralized project settings
# All paths, credentials, and dataset names are defined in
# core/settings.py to keep the project consistent.
# ============================================================
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "core.settings")

import django
django.setup()

print(f"Project root: {PROJECT_ROOT}")
print("Django settings loaded successfully.")

In [ ]:
# ============================================================
# STEP 1: Download the raw dataset from Kaggle
# This reuses the shared download script at:
#   apps/ai_engine/grading/download_dataset.py
# It downloads to: notebooks/dataset/raw/ (settings.GRADING_RAW_DIR)
# If the dataset already exists, it will be skipped automatically.
# ============================================================
from apps.ai_engine.grading.download_dataset import download_dataset

download_dataset()

In [ ]:
# ============================================================
# STEP 2: Check if trained model weights exist
# The trained YOLOv10 weights (best.pt) are needed for:
#   - Running the grading API (inference)
#   - Notebooks 03 & 04 (XAI visualization, grading pipeline)
#
# If you ONLY want to TEST (not train), you need this file.
# If you want to TRAIN from scratch, skip this — run 01 then 02.
# ============================================================
from django.conf import settings

WEIGHTS_PATH = settings.AI_GRADING_BEST_PT_PATH

if WEIGHTS_PATH.exists():
    size_mb = WEIGHTS_PATH.stat().st_size / (1024 * 1024)
    print(f"Trained model found: {WEIGHTS_PATH}")
    print(f"  File size: {size_mb:.1f} MB")
else:
    print(f"Trained model NOT found at: {WEIGHTS_PATH}")
    print()
    print("To get the trained weights, choose one option:")
    print("  Option A - Train from scratch:")
    print("    Run this notebook (01) then notebook 02 to train.")
    print("  Option B - Download pre-trained weights:")
    print(f"    Place the best.pt file at: {WEIGHTS_PATH}")

# 01. Data Annotation Script

Jupyter notebook to script auto annotation.

In [ ]:
import os
import shutil
import random
from pathlib import Path
from ultralytics import YOLO
from django.conf import settings

# ============================================================
# PATH CONFIGURATION
# IMPORTANT: These paths use Django settings (core/settings.py)
# to stay consistent with the rest of the project.
#   settings.GRADING_RAW_DIR       -> notebooks/dataset/raw/
#   settings.GRADING_PROCESSED_DIR -> notebooks/dataset/processed/
# ============================================================
RAW_DIR = settings.GRADING_RAW_DIR / "Fruit And Vegetable Diseases Dataset"
PROCESSED_DIR = settings.GRADING_PROCESSED_DIR

# Data split ratios
TRAIN_RATIO = 0.80  # 80% Training
VAL_RATIO = 0.15    # 15% Validation during training
TEST_RATIO = 0.05   # 5% Independent testing

def setup_directories():
    """Create the standard 3-folder structure for YOLO (train, val, test)."""
    for split in ['train', 'val', 'test']:
        (PROCESSED_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (PROCESSED_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def create_yaml(classes):
    """Auto-generate dataset.yaml including the test split path."""
    yaml_path = PROCESSED_DIR / "dataset.yaml"
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(f"path: {PROCESSED_DIR.resolve().as_posix()}\n")
        f.write("train: images/train\n")
        f.write("val: images/val\n")
        f.write("test: images/test\n\n")  # Include test split path
        f.write("names:\n")
        for idx, cls_name in enumerate(classes):
            f.write(f"  {idx}: {cls_name}\n")
    print(f" Config file created: {yaml_path}")

def process_data():
    setup_directories()

    print(" Loading YOLOv8n for auto-annotation...")
    auto_model = YOLO("yolov8n.pt")

    classes = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
    create_yaml(classes)

    total_images = 0
    missing_boxes = 0

    for class_id, class_name in enumerate(classes):
        class_dir = RAW_DIR / class_name
        images = [p for p in class_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

        if not images:
            continue

        print(f" Processing class [{class_id}] {class_name}: {len(images)} images...")
        random.shuffle(images)

        # Split data into 3 sets
        val_idx = int(len(images) * TRAIN_RATIO)
        test_idx = val_idx + int(len(images) * VAL_RATIO)

        train_imgs = images[:val_idx]
        val_imgs = images[val_idx:test_idx]
        test_imgs = images[test_idx:]

        # Iterate over all 3 splits
        splits = [
            (train_imgs, 'train'),
            (val_imgs, 'val'),
            (test_imgs, 'test')
        ]

        for img_list, split_name in splits:
            for img_path in img_list:
                results = auto_model(img_path, verbose=False)
                boxes = results[0].boxes.xywhn

                new_img_path = PROCESSED_DIR / 'images' / split_name / f"{class_name}_{img_path.name}"
                shutil.copy(img_path, new_img_path)

                label_path = PROCESSED_DIR / 'labels' / split_name / f"{class_name}_{img_path.stem}.txt"
                with open(label_path, 'w') as f:
                    if len(boxes) > 0:
                        for box in boxes.tolist():
                            f.write(f"{class_id} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")
                    else:
                        # Fallback when no object detected
                        box = [0.5, 0.5, 0.9, 0.9]
                        missing_boxes += 1
                        f.write(f"{class_id} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")

                total_images += 1

    print("\n" + "="*50)
    print(f" DATA PROCESSING COMPLETE!")
    print(f"Total images processed: {total_images}")
    print(f"Images using fallback box: {missing_boxes}")
    print("="*50)

if __name__ == "__main__":
    process_data()